# Provider Walkthrough

Manual, step-by-step experimentation with `LocalOllamaProvider` (`backend/app/core/providers.py`) against your local Ollama instance. Run cells top to bottom, or jump into a section once **Setup** has run.

Covers what's implemented so far (the provider layer): checking Ollama is reachable, a single generation call, the tokens/sec speed test, and the two boundary-condition errors. More sections will be added here as later phases (registry, two-pass run loop, scorers) land.

## 1. Setup

In [ ]:
from app.core.config import settings
from app.core.providers import (
    LocalOllamaProvider,
    OllamaModelNotFoundError,
    OllamaUnreachableError,
)

MODEL = settings.DEFAULT_MODEL  # change to a model you have pulled locally
provider = LocalOllamaProvider()

## 2. Check Ollama is reachable

In [ ]:
import httpx

print("OLLAMA_BASE_URL:", settings.OLLAMA_BASE_URL)
response = httpx.get(f"{settings.OLLAMA_BASE_URL}/api/tags", timeout=5.0)
response.raise_for_status()
pulled_models = [m["name"] for m in response.json().get("models", [])]
print("Locally pulled models:", pulled_models)

## 3. Generate a response

In [ ]:
PROMPT = "Write a two-sentence summary of why the sky is blue."

result = await provider.generate_response(MODEL, PROMPT)
print(result.text)
print()
print("prompt_tokens:", result.prompt_tokens, "| completion_tokens:", result.completion_tokens)
print("total_duration_ns:", result.total_duration_ns, "| eval_duration_ns:", result.eval_duration_ns)

## 4. Speed test

In [ ]:
speed = await provider.test_speed(MODEL)
print(f"{speed.tokens_per_second:.2f} tokens/sec "
      f"({speed.completion_tokens} tokens in {speed.eval_duration_ns / 1e9:.2f}s)")

## 5. Error handling

The two boundary conditions `LocalOllamaProvider` raises explicit exceptions for.

### Ollama unreachable

In [ ]:
unreachable_provider = LocalOllamaProvider(base_url="http://localhost:9999")
try:
    await unreachable_provider.generate_response(MODEL, "hi")
except OllamaUnreachableError as e:
    print("Got expected OllamaUnreachableError:", e)

### Model not found

In [ ]:
try:
    await provider.generate_response("definitely-not-a-real-model:latest", "hi")
except OllamaModelNotFoundError as e:
    print("Got expected OllamaModelNotFoundError:", e)